# Phase 3: Prompt Engineering

**Goal:** Learn how to steer model output by crafting the right input — without changing the model itself.

You now know:
- How models process text (Phases 1, 1b)
- How they generate text and the role of temperature/sampling (Phase 2)

**This phase:** The input side — how the **prompt** you give the model dramatically changes the output.

We'll use **FLAN-T5** (an encoder-decoder model fine-tuned on instructions) for a practical task: **dialogue summarization**.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
import evaluate
import textwrap

# Load FLAN-T5 (small version — runs on CPU)
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.eval()

print(f"Model: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")
print(f"Type: Encoder-Decoder (reads full input, then generates output)")
# Helper defined up front so every comparison cell below can use it
def generate_flan(prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_length, do_sample=False)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Side-by-side: GPT-2 (base) vs FLAN-T5 (instruction-tuned)
from transformers import GPT2LMHeadModel, GPT2Tokenizer

gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_model.eval()

def generate_gpt2(prompt, max_new_tokens=50):
    inputs = gpt2_tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = gpt2_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        )
    return gpt2_tokenizer.decode(output[0], skip_special_tokens=True)

test_prompts = [
    "Translate to French: The cat is sleeping.",
    "What is the capital of Japan?",
    "Is the following sentence positive or negative? I hated this movie.",
]

print("Same prompts → two very different models:\n")
print(f"{'─'*90}")
for prompt in test_prompts:
    flan_response = generate_flan(prompt)
    gpt2_response = generate_gpt2(prompt)
    
    print(f"\nPrompt:   {prompt}")
    print(f"FLAN-T5:  {flan_response}")
    print(f"GPT-2:    {gpt2_response[:120]}...")
    print(f"{'─'*90}")

print("\nGPT-2 just continues the text — it doesn't understand you're asking it to DO something.")
print("FLAN-T5 was trained on instructions, so it actually performs the task.")
print("Same transformer architecture, but instruction tuning makes all the difference.")

---
## 1. Why FLAN-T5 Instead of GPT-2?

GPT-2 (Phase 2) is a **base model** — it just predicts the next token. It wasn't trained to follow instructions.

FLAN-T5 was **fine-tuned on thousands of instruction-following tasks**, so it actually understands prompts like "Summarize this" or "Translate to French."

| Model | Type | Follows instructions? | Best for |
|---|---|---|---|
| GPT-2 | Decoder-only, base | No — just continues text | Text completion |
| FLAN-T5 | Encoder-decoder, instruction-tuned | Yes | Summarization, translation, Q&A |

This is the difference between a **base model** and an **instruction-tuned model** — a concept you'll revisit in Phase 4 (fine-tuning).

In [ ]:
# Quick demo: FLAN-T5 follows instructions, GPT-2 doesn't

def generate_flan(prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=False,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_prompts = [
    "Translate to French: The cat is sleeping.",
    "What is the capital of Japan?",
    "Is the following sentence positive or negative? I hated this movie.",
]

print("FLAN-T5 following instructions:\n")
for prompt in test_prompts:
    response = generate_flan(prompt)
    print(f"  Prompt:   {prompt}")
    print(f"  Response: {response}\n")

print("It understands what you're asking — not just continuing text.")
print("This is the power of instruction tuning.")

---
## 2. The Task: Dialogue Summarization

We'll work with real dialogues and try to get the model to summarize them.

This is directly relevant to customer support — imagine summarizing a long customer-agent conversation into a brief ticket note.

In [ ]:
# Load the DialogSum dataset
dataset = load_dataset("knkarthick/dialogsum", split="test")

print(f"Dataset size: {len(dataset)} dialogues\n")
print(f"Columns: {dataset.column_names}\n")

# Look at a few examples
for i in range(3):
    example = dataset[i]
    print(f"{'='*60}")
    print(f"Example {i+1}:")
    print(f"\nDialogue:")
    print(textwrap.fill(example['dialogue'], width=70))
    print(f"\nHuman summary: {example['summary']}")
    print()

In [ ]:
# Pick a few examples we'll use throughout this notebook

examples = [dataset[i] for i in [0, 5, 10, 15, 20]]

# We'll focus on this one dialogue for most experiments
main_dialogue = examples[0]['dialogue']
main_summary = examples[0]['summary']

print("Main dialogue for experiments:")
print(f"\n{main_dialogue}")
print(f"\nReference summary: {main_summary}")

### GPT-4 Comparison: Does Prompt Wording Matter on a Large Model?

FLAN-T5-base showed similar results across prompt styles. Let's see if GPT-4o-mini shows a bigger difference — larger models are more sensitive to prompt nuances.


In [ ]:
# Load OpenAI credentials from .env file
from dotenv import load_dotenv
load_dotenv("../.env")

import os
from openai import OpenAI

openai_key = os.environ.get("OPENAI_API_KEY", "")

if openai_key and not openai_key.startswith("sk-your"):
    gpt4_client = OpenAI(api_key=openai_key)
    gpt4_model = "gpt-4o-mini"
    gpt4_ready = True
    print(f"OpenAI connected")
    print(f"Model: {gpt4_model}")
else:
    gpt4_ready = False
    print("⚠ OPENAI_API_KEY not found in .env file.")
    print("  Add your key to ../.env to run this section.")


In [ ]:
# Compare prompt styles: FLAN-T5 vs GPT-4 on the same dialogue

if not gpt4_ready:
    print("Skipping — no GPT-4 credentials. See cell above.")
else:
    def generate_gpt4(prompt):
        resp = gpt4_client.chat.completions.create(
            model=gpt4_model,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=500,
        )
        return resp.choices[0].message.content or ""

    zero_shot_prompts = {
        "Bare minimum": f"{main_dialogue}\n\nSummary:",
        "Simple instruction": f"Summarize the following dialogue:\n\n{main_dialogue}",
        "Detailed instruction": (
            f"Summarize the following dialogue in 1-2 sentences. "
            f"Focus on the key outcome of the conversation.\n\n{main_dialogue}"
        ),
        "Structured output": (
            f"Read the following dialogue and provide:\n"
            f"- Topic: (main topic)\n"
            f"- Summary: (1 sentence summary)\n\n"
            f"Dialogue:\n{main_dialogue}"
        ),
    }

    print(f"Reference: {main_summary}\n")
    print(f"{'Prompt Style':<25} {'FLAN-T5':<50} {'GPT-4'}")
    print("─" * 130)

    for i, (name, prompt) in enumerate(zero_shot_prompts.items(), 1):
        print(f"[{i}/{len(zero_shot_prompts)}] Testing '{name}'...", flush=True)
        flan_result = generate_flan(prompt)
        gpt4_result = generate_gpt4(prompt)
        flan_short = flan_result[:47] + "..." if len(flan_result) > 50 else flan_result
        gpt4_short = gpt4_result[:60] + "..." if len(gpt4_result) > 60 else gpt4_result
        
        print(f"\n{name}:")
        print(f"  FLAN-T5: {flan_result}")
        print(f"  GPT-4:   {gpt4_result}")
        print()

    print("─" * 130)
    print("\nNotice: GPT-4 shows more variation between prompt styles,")
    print("especially for 'Structured output' where it follows the format precisely.")
    print("Larger models are more sensitive to prompt wording — prompt engineering matters MORE.")

---
## 3. Zero-Shot Prompting: No Examples, Just Instructions

**Zero-shot** = give the model a task description and input, with no examples of how to do it.

The model relies entirely on its pre-training and instruction tuning to figure out what you want.

Let's try different zero-shot prompts on the same dialogue and see how the wording affects quality.

In [ ]:
# Try multiple zero-shot prompt styles

zero_shot_prompts = {
    "Bare minimum": f"{main_dialogue}\n\nSummary:",
    
    "Simple instruction": f"Summarize the following dialogue:\n\n{main_dialogue}",
    
    "Detailed instruction": f"Summarize the following dialogue in 1-2 sentences. "
        f"Focus on the key outcome of the conversation.\n\n{main_dialogue}",
    
    "Role-based": f"You are a customer support manager. Summarize this conversation "
        f"into a brief ticket note:\n\n{main_dialogue}",
    
    "Structured output": f"Read the following dialogue and provide:\n"
        f"- Topic: (main topic)\n"
        f"- Summary: (1 sentence summary)\n\n"
        f"Dialogue:\n{main_dialogue}",
}

print(f"Reference summary: {main_summary}\n")
print("=" * 60)

for name, prompt in zero_shot_prompts.items():
    response = generate_flan(prompt)
    print(f"\n{name}:")
    print(f"  → {response}")

print(f"\n{'='*60}")
print("\nNotice how the SAME model gives different outputs based on")
print("how you phrase the instruction. The model didn't change — only the prompt did.")
print("This is prompt engineering.")

### YOUR UNDERSTANDING

*Which zero-shot prompt style gave the best result? Why do you think the wording matters?*

> (your notes here)


---
## 4. One-Shot Prompting: Show One Example

**One-shot** = give the model one example of input → output before asking it to do the task.

This teaches the model the **format and style** you expect.

In [ ]:
# One-shot: provide one example, then the actual task

example_dialogue = examples[1]['dialogue']
example_summary = examples[1]['summary']

one_shot_prompt = f"""Summarize the following dialogues in 1-2 sentences.

Dialogue:
{example_dialogue}

Summary: {example_summary}

Dialogue:
{main_dialogue}

Summary:"""

# Compare zero-shot vs one-shot
zero_shot_prompt = f"Summarize the following dialogue in 1-2 sentences:\n\n{main_dialogue}"

zero_result = generate_flan(zero_shot_prompt)
one_result = generate_flan(one_shot_prompt, max_length=150)

print(f"Reference: {main_summary}\n")
print(f"Zero-shot: {zero_result}")
print(f"One-shot:  {one_result}")
print(f"\nThe example teaches the model what style and length you expect.")

---
## 5. Few-Shot Prompting: Show Multiple Examples

**Few-shot** = give 2-5 examples before the actual task.

More examples → the model better understands the pattern. But there's a trade-off: more examples = more tokens = less room for the actual input (context window limit).

In [ ]:
# Few-shot: provide multiple examples

def build_few_shot_prompt(n_examples, target_dialogue):
    prompt = "Summarize each dialogue in 1-2 concise sentences.\n"
    
    for i in range(n_examples):
        ex = examples[i + 1]  # skip examples[0] since that's our target
        prompt += f"\nDialogue:\n{ex['dialogue']}\n"
        prompt += f"Summary: {ex['summary']}\n"
    
    prompt += f"\nDialogue:\n{target_dialogue}\n"
    prompt += "Summary:"
    
    return prompt

print(f"Reference: {main_summary}\n")
print(f"{'Shots':<8} {'Tokens':<10} {'Output'}")
print("-" * 70)

for n in [0, 1, 2, 3]:
    if n == 0:
        prompt = f"Summarize the dialogue in 1-2 concise sentences.\n\nDialogue:\n{main_dialogue}\nSummary:"
    else:
        prompt = build_few_shot_prompt(n, main_dialogue)
    
    token_count = len(tokenizer.encode(prompt))
    result = generate_flan(prompt, max_length=150)
    print(f"{n}-shot{'':<3} {token_count:<10} {result}")

print(f"\nNotice: more examples = more tokens used on the prompt itself.")
print(f"FLAN-T5-base has a 512 token limit — few-shot can hit that fast.")
print(f"Larger models (GPT-4: 128K tokens) have much more room.")

### YOUR UNDERSTANDING

*What's the trade-off between zero-shot, one-shot, and few-shot prompting?*

> (your notes here)


---
## 6. Measuring Quality: ROUGE Scores

So far we've been eyeballing quality. Let's measure it properly.

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) compares generated summaries against reference summaries:

| Metric | What it measures |
|---|---|
| **ROUGE-1** | Overlap of individual words (unigrams) |
| **ROUGE-2** | Overlap of two-word phrases (bigrams) |
| **ROUGE-L** | Longest common subsequence |

Scores range from 0 to 1 — higher is better.

In [ ]:
# Load ROUGE evaluator
rouge = evaluate.load("rouge")

# Evaluate different prompting strategies on multiple dialogues
test_examples = [dataset[i] for i in range(10)]

strategies = {
    "Bare (just dialogue + Summary:)": lambda d: f"{d}\n\nSummary:",
    "Zero-shot instruction": lambda d: f"Summarize the following dialogue in 1-2 sentences:\n\n{d}",
    "Detailed zero-shot": lambda d: f"Summarize the following dialogue in 1-2 sentences. Focus on the key outcome and who is involved.\n\n{d}",
}

print(f"Evaluating on {len(test_examples)} dialogues...\n")

for strategy_name, prompt_fn in strategies.items():
    predictions = []
    references = []
    
    for ex in test_examples:
        prompt = prompt_fn(ex['dialogue'])
        pred = generate_flan(prompt)
        predictions.append(pred)
        references.append(ex['summary'])
    
    scores = rouge.compute(predictions=predictions, references=references)
    print(f"{strategy_name}:")
    print(f"  ROUGE-1: {scores['rouge1']:.4f}")
    print(f"  ROUGE-2: {scores['rouge2']:.4f}")
    print(f"  ROUGE-L: {scores['rougeL']:.4f}")
    print()

In [ ]:
# See specific examples: prediction vs reference

prompt_fn = lambda d: f"Summarize the following dialogue in 1-2 sentences. Focus on the key outcome.\n\n{d}"

print("Sample predictions vs references:\n")
for i, ex in enumerate(test_examples[:5]):
    pred = generate_flan(prompt_fn(ex['dialogue']))
    ref = ex['summary']
    
    scores = rouge.compute(predictions=[pred], references=[ref])
    
    print(f"Example {i+1} (ROUGE-1: {scores['rouge1']:.3f}):")
    print(f"  Reference:  {ref}")
    print(f"  Predicted:  {pred}")
    print()

### YOUR UNDERSTANDING

*Why is ROUGE useful but not perfect as a metric for summarization?*

> (your notes here)


---
## 7. Prompt Engineering Techniques

Beyond zero/one/few-shot, there are specific techniques that improve output quality.

In [ ]:
# Technique 1: Be specific about format

dialogue = test_examples[2]['dialogue']

vague = f"Summarize:\n{dialogue}"
specific = f"Summarize the following dialogue in exactly one sentence, starting with the names of the people involved:\n\n{dialogue}"

print(f"Vague prompt:    {generate_flan(vague)}")
print(f"Specific prompt: {generate_flan(specific)}")
print(f"\nSpecificity in the prompt → specificity in the output.")

In [ ]:
# Technique 2: Output constraints

dialogue = test_examples[3]['dialogue']

unconstrained = f"Summarize this dialogue:\n\n{dialogue}"
constrained = f"Summarize this dialogue in no more than 15 words:\n\n{dialogue}"

result_1 = generate_flan(unconstrained)
result_2 = generate_flan(constrained)

print(f"Unconstrained ({len(result_1.split())} words): {result_1}")
print(f"Constrained ({len(result_2.split())} words):   {result_2}")
print(f"\nThe model tries to respect length constraints (not always perfectly).")

In [ ]:
# Technique 3: Chain-of-thought — ask the model to reason step by step
# (works better on larger models, but let's see what FLAN-T5 does)

dialogue = test_examples[4]['dialogue']

direct = f"What is the main topic of this dialogue?\n\n{dialogue}"

chain_of_thought = f"""Read the following dialogue. First identify who is speaking and what each person wants. Then determine the main topic.

{dialogue}

Main topic:"""

print(f"Direct:           {generate_flan(direct)}")
print(f"Chain-of-thought: {generate_flan(chain_of_thought, max_length=150)}")
print(f"\nChain-of-thought prompting guides the model through reasoning steps.")
print(f"More effective on larger models (GPT-4, Claude) than small ones like FLAN-T5-base.")

In [ ]:
# Technique 4: Negative instructions — tell the model what NOT to do

dialogue = test_examples[0]['dialogue']

without_negative = f"Summarize this dialogue:\n\n{dialogue}"

with_negative = f"""Summarize this dialogue in 1-2 sentences. 
Do NOT include specific names. 
Do NOT copy phrases directly from the dialogue.

{dialogue}"""

print(f"Without constraints: {generate_flan(without_negative)}")
print(f"With negatives:     {generate_flan(with_negative, max_length=150)}")
print(f"\nNegative instructions help prevent common failure modes.")

---
## 8. The Limits of Prompt Engineering

Prompt engineering is powerful, but it has clear limits. Let's see them.

In [ ]:
# Limit 1: The model's base capability is the ceiling
# No prompt can make a small model do what it can't do

hard_task = """Analyze the following dialogue. Provide:
1. Sentiment of each speaker (positive/negative/neutral)
2. Urgency level (low/medium/high)
3. Recommended action
4. One-sentence summary

Dialogue:
#Person1#: I need to return this laptop. It stopped working after 2 days!
#Person2#: I'm sorry to hear that. Let me check your order.
#Person1#: This is unacceptable. I want a full refund immediately.
#Person2#: I understand your frustration. I'll process that right away."""

result = generate_flan(hard_task, max_length=200)
print(f"Complex prompt result:")
print(f"  {result}")
print(f"\nFLAN-T5-base struggles with complex multi-part instructions.")
print(f"This is where fine-tuning (Phase 4) or larger models help.")
print(f"Prompt engineering works within the model's existing capabilities.")

In [ ]:
# Limit 2: Context window
# You can only fit so much into the prompt

long_dialogue = "\n".join([ex['dialogue'] for ex in test_examples[:5]])
prompt = f"Summarize all of the following dialogues:\n\n{long_dialogue}"

token_count = len(tokenizer.encode(prompt))
print(f"Prompt length: {token_count} tokens")
print(f"FLAN-T5-base limit: 512 tokens")
print(f"Exceeds limit: {token_count > 512}")
print(f"\nWhen your prompt exceeds the context window, the model either:")
print(f"  - Truncates (loses information)")
print(f"  - Errors out")
print(f"\nThis is why RAG (Phase 5) exists — to retrieve only the relevant")
print(f"context instead of stuffing everything into the prompt.")

---
## 9. Prompt Engineering Cheat Sheet

| Technique | What it does | Example |
|---|---|---|
| **Zero-shot** | Task instruction only | "Summarize this dialogue" |
| **One-shot** | One example + task | Example pair → "Now summarize this" |
| **Few-shot** | Multiple examples + task | 3-5 example pairs → "Now summarize this" |
| **Be specific** | Constrain the format | "In exactly one sentence, starting with..." |
| **Length control** | Limit output size | "In no more than 15 words" |
| **Chain-of-thought** | Ask for step-by-step reasoning | "First identify X, then determine Y" |
| **Negative instructions** | Say what NOT to do | "Do NOT include names" |
| **Role assignment** | Give the model a persona | "You are a customer support manager" |

### When Prompt Engineering Isn't Enough

| Problem | Solution |
|---|---|
| Model doesn't know your domain data | **RAG** (Phase 5) — retrieve relevant docs |
| Model doesn't follow your format consistently | **Fine-tuning** (Phase 4) — train on your data |
| Model isn't accurate enough for your task | **Fine-tuning** or use a **larger model** |
| Prompt is too long for context window | **RAG** — only retrieve what's needed |

### YOUR UNDERSTANDING — Final Reflection

*In your own words, what is prompt engineering and why is it the first thing to try before fine-tuning?*

> (your notes here)

*When would prompt engineering NOT be sufficient? Give a concrete example.*

> (your notes here)

*How does this connect to the customer support project? What role will prompts play?*

> (your notes here)


---
## Next Up: Phase 4 — Fine-Tuning with LoRA/PEFT

Prompt engineering steers the model without changing it. Fine-tuning actually **changes the model's weights** to specialize it for your task. You'll:
- Fine-tune FLAN-T5 on dialogue summarization
- Compare full fine-tuning vs LoRA (lightweight fine-tuning)
- Measure improvement with ROUGE scores
- See catastrophic forgetting firsthand

**Before moving on**, fill in all `YOUR UNDERSTANDING` sections and experiment with different prompts!